# GridCast Europe – Q-Phase: Fragestellung und Experimentdesign

**QUA³CK-Phase:** Q – *Question*  
**Fassung:** 25.07.2026  
**Projekt-Pivot:** WasteWise → GridCast Europe am 22.07.2026  
**Prüfungstermin:** 28.07.2026

Dieses Notebook definiert, **welche Fragen GridCast Europe beantwortet**, welche
Hypothesen geprüft werden und anhand welcher Regeln das spätere Ergebnis als
erfolgreich oder nicht erfolgreich gilt.

Es ist das methodische Lastenheft für:

- [`01_data_import_merge_eda.ipynb`](01_data_import_merge_eda.ipynb) – U-Phase,
- `02_a3_model_development.ipynb` – A³-Phase,
- die abschließende C-Phase und
- die Streamlit-Anwendung in der K-Phase.


## Transparenz zum nachgezogenen Q-Notebook

Das ursprünglich vorhandene Q-Dokument gehörte zur verworfenen
**WasteWise-Idee**. Nach dem Pivot zu GridCast Europe wurde zunächst die neue
Forschungsrichtung im Repository festgelegt und anschließend die Datenpipeline
End-to-End geprüft. Dieses GridCast-Q-Notebook wird deshalb am 25.07.2026
**nachgezogen**.

Damit kein falscher chronologischer Eindruck entsteht, gelten folgende Regeln:

1. Der Pivot und seine Gründe werden ausdrücklich dokumentiert.
2. Forschungsrichtung, Vergleichslogik und Leakage-Schutz werden als
   verbindliche Spezifikation festgehalten.
3. Bereits bekannte U-Ergebnisse werden nur in einem klar markierten
   Rückmeldeabschnitt genannt.
4. Hypothesen werden nicht passend zu späteren Modellresultaten umformuliert.
5. Das Testjahr 2019 bleibt bis zur C-Phase unangetastet.

Das Notebook rekonstruiert somit transparent die Q-Phase des neuen Projekts,
statt das alte WasteWise-Notebook fälschlich als GridCast-Dokument auszugeben.


In [1]:
from __future__ import annotations

import pandas as pd

try:
    from IPython.display import Markdown
except ImportError:
    class Markdown(str):
        pass

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

PROJECT_SPEC = {
    "project": "GridCast Europe",
    "phase": "Q – Question",
    "pivot_date": "2026-07-22",
    "notebook_date": "2026-07-25",
    "exam_date": "2026-07-28",
    "countries": ("DE", "FR", "PL"),
    "observation_unit": "Land × UTC-Stunde",
    "target": "Tatsächliche nationale Stromlast in MW",
    "train": "2015-01-01 bis 2017-12-31",
    "validation": "2018-01-01 bis 2018-12-31",
    "test": "2019-01-01 bis 2019-12-31",
    "primary_metric": "Makro-nMAE",
    "baselines": ("Ländermittelwert", "Kalender-Baseline"),
    "model_families": ("Regularisierte lineare Regression", "Gradient Boosting"),
    "core_uses_load_lags": False,
}

display(Markdown(
    f"**Aktive Projektspezifikation:** {PROJECT_SPEC['project']} · "
    f"{PROJECT_SPEC['observation_unit']} · Länder "
    f"{', '.join(PROJECT_SPEC['countries'])}"
))


**Aktive Projektspezifikation:** GridCast Europe · Land × UTC-Stunde · Länder DE, FR, PL

## 1. Entwicklungsverlauf und Projekt-Pivot

### Ausgangslage: WasteWise

WasteWise sollte Bildklassifikation und eine standortabhängige Prognose der
Verwertungswahrscheinlichkeit verbinden. Für den Prüfungszeitraum erwies sich
dieser Ansatz als fachlich und zeitlich überdimensioniert:

- keine unmittelbar verfügbare, belastbare Zielvariable,
- unklare Joinbarkeit und semantische Vergleichbarkeit der Datensätze,
- Mapping zwischen Abfallklassen, Regionen und Verwertungsdaten,
- zwei getrennte Trainings- und Evaluationspipelines,
- hohes Risiko, viel Integrationsarbeit ohne prüfbares Endergebnis zu erzeugen.

### Entscheidung: GridCast Europe

GridCast Europe verwendet zwei fachlich zusammenpassende, stündliche
OPSD-Datensätze. Die Zielvariable ist eindeutig, der zeitliche Join ist
prüfbar und die Modellgüte lässt sich über einen chronologischen Backtest
objektiv messen.

Der Wechsel ist keine inhaltliche Kosmetik, sondern eine dokumentierte
Scope-Korrektur zugunsten eines vollständigen, nachvollziehbaren
Data-Science-Projekts.


In [2]:
timeline = pd.DataFrame(
    [
        {
            "Zeitpunkt": "bis 21.07.2026",
            "Stand": "WasteWise",
            "Entscheidung / Ergebnis": (
                "Zweistufige Bild- und Tabular-Pipeline konzipiert; "
                "Datenjoin und Zielvariable nicht belastbar genug."
            ),
        },
        {
            "Zeitpunkt": "22.07.2026",
            "Stand": "Projekt-Pivot",
            "Entscheidung / Ergebnis": (
                "Wechsel zu GridCast Europe: stündliche Stromlast als "
                "eindeutige Regressionszielvariable."
            ),
        },
        {
            "Zeitpunkt": "23.–25.07.2026",
            "Stand": "Q- und U-Entscheidungen",
            "Entscheidung / Ergebnis": (
                "Forschungsrichtung, Baselines, chronologischer Split, "
                "OPSD-Join und Kernländer festgelegt beziehungsweise geprüft."
            ),
        },
        {
            "Zeitpunkt": "25.07.2026",
            "Stand": "GridCast-Q-Notebook",
            "Entscheidung / Ergebnis": (
                "Fragestellung, Hypothesen und Erfolgskriterien transparent "
                "in Notebook-Form zusammengeführt."
            ),
        },
    ]
)

display(timeline.set_index("Zeitpunkt"))


,Stand,Entscheidung / Ergebnis
Zeitpunkt,,
bis 21.07.2026,WasteWise,Zweistufige Bild- und Tabular-Pipeline konzipiert; Datenjoin und Zielvariable nicht belastbar genug.
22.07.2026,Projekt-Pivot,Wechsel zu GridCast Europe: stündliche Stromlast als eindeutige Regressionszielvariable.
23.–25.07.2026,Q- und U-Entscheidungen,"Forschungsrichtung, Baselines, chronologischer Split, OPSD-Join und Kernländer festgelegt beziehungsweise geprüft."
25.07.2026,GridCast-Q-Notebook,"Fragestellung, Hypothesen und Erfolgskriterien transparent in Notebook-Form zusammengeführt."


## 2. Problem, Zielgruppe und Nutzen

### Problem

Nationale Stromlast folgt wiederkehrenden Tages-, Wochen- und Saisonmustern,
reagiert aber zusätzlich auf Wetter und länderspezifische Strukturen. Ein
Modell ist nur dann nützlich, wenn es nicht lediglich diese groben Mittelwerte
reproduziert, sondern gegenüber einer starken Kalender-Baseline einen
messbaren Zusatznutzen zeigt.

### Primäre Zielgruppen

| Zielgruppe | Nutzen |
|---|---|
| Energieanalysten und Planer | historische Lastmuster, Modellfehler und transparente Szenarioeffekte untersuchen |
| Lehrende und Studierende | vollständigen QUA³CK-Prozess reproduzierbar nachvollziehen |
| Fachlich interessierte Öffentlichkeit | Unterschiede zwischen Prognose, Baseline und Was-wäre-wenn-Szenario verstehen |

### Bewusste Nicht-Zielgruppe

GridCast Europe ist **kein operatives Netzleitsystem**. Die Anwendung trifft
keine Echtzeitentscheidungen, erstellt keine konkrete Wettervorhersage und
behauptet keine Blackout-Wahrscheinlichkeit.


## 3. Forschungsfragen

### Primärfrage

> **Wie genau lässt sich die stündliche Stromlast ausgewählter europäischer
> Länder anhand historischer Last-, Wetter- und Kalenderdaten für einen
> chronologisch späteren, vollständig zurückgehaltenen Zeitraum
> prognostizieren?**

Die Primärfrage untersucht zeitliche Generalisierung. Ein fester operativer
Day-ahead-Horizont ist nicht Bestandteil des Pflichtumfangs.

### Erweiterte Streamlit-Frage

> **Wie verändert sich ein aus historischen Mustern abgeleitetes Lastprofil
> für einen frei wählbaren Zukunftszeitpunkt unter klimatologischen und
> strukturellen Szenarioannahmen?**

Diese zweite Frage beschreibt eine **konditionale Projektion**. Sie ist weder
eine konkrete Wettervorhersage noch eine autonome Langfristprognose der realen
Stromnachfrage.

### Unterfragen

1. Sind Last und Wetter auf `Land × UTC-Stunde` hinreichend vollständig und
   1:1 joinbar?
2. Wie viel Erklärungskraft liefern Kalenderstrukturen bereits ohne ML?
3. Verbessern Wettermerkmale die Prognose gegenüber einer Kalender-Baseline?
4. Profitieren die Daten von nichtlinearen Modellen?
5. Wie unterscheiden sich Fehler nach Land, Tageszeit und Saison?


## 4. Hypothesen und vorab definierte Entscheidungsregeln

Die Hypothesen sind ergebnisoffen. Auch eine nicht bestätigte Hypothese ist ein
fachlich valides Resultat, sofern Experiment und Auswertung korrekt sind.


In [3]:
hypotheses = pd.DataFrame(
    [
        {
            "ID": "H1 – Daten-Eignung",
            "Hypothese": (
                "Last- und Wetterdaten sind für mindestens drei Länder im "
                "gemeinsamen Modellfenster stündlich 1:1 joinbar."
            ),
            "Entscheidungsregel": (
                "≥ 95 % beobachtete Zielwerte je Kernland, eindeutige Schlüssel "
                "und ausreichend vollständige Wettermerkmale."
            ),
            "Geprüft in": "U",
        },
        {
            "ID": "H2 – Kalenderstruktur",
            "Hypothese": (
                "Die Kalender-Baseline ist genauer als der konstante "
                "länderspezifische Mittelwert."
            ),
            "Entscheidungsregel": (
                "Niedrigerer Makro-nMAE auf dem jeweiligen Out-of-sample-Zeitraum."
            ),
            "Geprüft in": "A³ / C",
        },
        {
            "ID": "H3 – ML-Mehrwert",
            "Hypothese": (
                "Das final ausgewählte ML-Modell prognostiziert genauer als "
                "die Kalender-Baseline."
            ),
            "Entscheidungsregel": (
                "Makro-nMAE 2019 unter Kalender-Baseline; ≥ 5 % relative "
                "Verbesserung ist das praktische Ziel."
            ),
            "Geprüft in": "C",
        },
        {
            "ID": "H4 – Nichtlinearität",
            "Hypothese": (
                "Gradient Boosting bildet nichtlineare Kalender-Wetter-"
                "Zusammenhänge besser ab als regularisierte lineare Regression."
            ),
            "Entscheidungsregel": (
                "Niedrigerer Makro-nMAE auf der Validierung 2018; "
                "2019 darf für diese Auswahl nicht verwendet werden."
            ),
            "Geprüft in": "A³",
        },
        {
            "ID": "H5 – Wettermehrwert",
            "Hypothese": (
                "Kalender- plus Wetterfeatures sind genauer als ein "
                "vergleichbares Modell nur mit Land und Kalenderfeatures."
            ),
            "Entscheidungsregel": (
                "Positiver Makro-nMAE-Gewinn in einer Ablation auf 2018."
            ),
            "Geprüft in": "A³",
        },
    ]
)

display(hypotheses.set_index("ID"))


,Hypothese,Entscheidungsregel,Geprüft in
ID,,,
H1 – Daten-Eignung,Last- und Wetterdaten sind für mindestens drei Länder im gemeinsamen Modellfenster stündlich 1:1 joinbar.,"≥ 95 % beobachtete Zielwerte je Kernland, eindeutige Schlüssel und ausreichend vollständige Wettermerkmale.",U
H2 – Kalenderstruktur,Die Kalender-Baseline ist genauer als der konstante länderspezifische Mittelwert.,Niedrigerer Makro-nMAE auf dem jeweiligen Out-of-sample-Zeitraum.,A³ / C
H3 – ML-Mehrwert,Das final ausgewählte ML-Modell prognostiziert genauer als die Kalender-Baseline.,Makro-nMAE 2019 unter Kalender-Baseline; ≥ 5 % relative Verbesserung ist das praktische Ziel.,C
H4 – Nichtlinearität,Gradient Boosting bildet nichtlineare Kalender-Wetter-Zusammenhänge besser ab als regularisierte lineare Regression.,Niedrigerer Makro-nMAE auf der Validierung 2018; 2019 darf für diese Auswahl nicht verwendet werden.,A³
H5 – Wettermehrwert,Kalender- plus Wetterfeatures sind genauer als ein vergleichbares Modell nur mit Land und Kalenderfeatures.,Positiver Makro-nMAE-Gewinn in einer Ablation auf 2018.,A³


## 5. Erfolgsmetriken

Absolute Fehler in MW werden benötigt, sind zwischen unterschiedlich großen
Stromsystemen aber nicht direkt vergleichbar. Deshalb wird zusätzlich pro Land
normalisiert und anschließend **gleichgewichtet über die Länder gemittelt**.

Für Land $c$:

$$
\mathrm{MAE}_c
=
\frac{1}{n_c}
\sum_{i=1}^{n_c}
\left|y_i-\widehat y_i\right|
$$

$$
\mathrm{nMAE}_c
=
\frac{\mathrm{MAE}_c}
{\overline{|y_c|}}
$$

Primäre Auswahlmetrik:

$$
\mathrm{Makro\text{-}nMAE}
=
\frac{1}{|C|}
\sum_{c \in C}\mathrm{nMAE}_c
$$

Relative Verbesserung gegenüber der Kalender-Baseline:

$$
\mathrm{Verbesserung}_{\%}
=
100
\left(
1-
\frac{\mathrm{Makro\text{-}nMAE}_{Modell}}
{\mathrm{Makro\text{-}nMAE}_{Kalender}}
\right)
$$

- **Formaler Mindestnachweis:** Verbesserung $>0\,\%$
- **Praktisches Ziel:** Verbesserung $\geq 5\,\%$
- **Zusätzlich:** MAE, RMSE, sMAPE und $R^2$
- **Pflichtdiagnostik:** Kennzahlen je Land, Tageszeit und Saison

Das praktische 5-%-Ziel ist bewusst anspruchsvoller als die formale Aussage,
dass überhaupt ein zusätzlicher ML-Mehrwert nachgewiesen wurde.


In [4]:
success_criteria = pd.DataFrame(
    [
        {
            "Kriterium": "Reproduzierbarkeit",
            "Mindestanforderung": "Import, Join, Features und Splits vollständig per Code ausführbar",
            "Rolle": "Pflicht",
        },
        {
            "Kriterium": "Datenqualität",
            "Mindestanforderung": "H1 erfüllt; keine ungeklärten Schlüssel- oder Definitionsprobleme",
            "Rolle": "Pflicht",
        },
        {
            "Kriterium": "ML-Mehrwert",
            "Mindestanforderung": "Makro-nMAE des finalen Modells unter der Kalender-Baseline",
            "Rolle": "Primäres Erfolgskriterium",
        },
        {
            "Kriterium": "Praktische Relevanz",
            "Mindestanforderung": "Mindestens 5 % Verbesserung gegenüber der Kalender-Baseline",
            "Rolle": "Zielwert",
        },
        {
            "Kriterium": "Transparente Grenzen",
            "Mindestanforderung": "Backtest und Zukunftsszenario sprachlich und technisch getrennt",
            "Rolle": "Pflicht",
        },
        {
            "Kriterium": "Präsentierbarkeit",
            "Mindestanforderung": "Fehler und Szenarioannahmen in der App verständlich sichtbar",
            "Rolle": "Pflicht",
        },
    ]
)

display(success_criteria.set_index("Kriterium"))


,Mindestanforderung,Rolle
Kriterium,,
Reproduzierbarkeit,"Import, Join, Features und Splits vollständig per Code ausführbar",Pflicht
Datenqualität,H1 erfüllt; keine ungeklärten Schlüssel- oder Definitionsprobleme,Pflicht
ML-Mehrwert,Makro-nMAE des finalen Modells unter der Kalender-Baseline,Primäres Erfolgskriterium
Praktische Relevanz,Mindestens 5 % Verbesserung gegenüber der Kalender-Baseline,Zielwert
Transparente Grenzen,Backtest und Zukunftsszenario sprachlich und technisch getrennt,Pflicht
Präsentierbarkeit,Fehler und Szenarioannahmen in der App verständlich sichtbar,Pflicht


## 6. Daten- und Evaluationsplan

### Geplante Kernquellen

| Quelle | Rolle | Verbindung |
|---|---|---|
| OPSD Time Series, Version 2020-10-06 | tatsächliche nationale Stromlast in MW | Ländercode + UTC-Stunde |
| OPSD Weather Data, Version 2020-09-16 | Temperatur und Strahlung | Ländercode + UTC-Stunde |

Beobachtungseinheit:

> **Land × UTC-Stunde**

### Räumlicher Scope

- Deutschland (`DE`)
- Frankreich (`FR`)
- Polen (`PL`)

Weitere Länder bleiben technisch ergänzbar, benötigen aber denselben
Qualitäts-, Plausibilitäts- und Evaluationsprozess.

### Chronologischer Split

| Block | Zeitraum | Verwendung |
|---|---|---|
| Training | 2015–2017 | Parameter, Preprocessing und Baseline-Durchschnitte lernen |
| Validierung | 2018 | Features, Modellklasse und Hyperparameter auswählen |
| Test | 2019 | eingefrorene Konfiguration genau einmal abschließend bewerten |

Ein zufälliger Split wäre für diese zeitliche Forschungsfrage ungeeignet, weil
er zukünftige und vergangene Beobachtungen vermischen könnte.


In [5]:
experiment_blocks = pd.DataFrame(
    [
        {
            "Block": "Training",
            "Jahre": "2015–2017",
            "Darf beeinflussen": "Preprocessing, Modellparameter, Baselines",
            "Darf nicht beeinflussen": "keine Einschränkung innerhalb des Trainings",
        },
        {
            "Block": "Validierung",
            "Jahre": "2018",
            "Darf beeinflussen": "Featurewahl, Modellklasse, Hyperparameter",
            "Darf nicht beeinflussen": "finale Testkennzahl",
        },
        {
            "Block": "Test",
            "Jahre": "2019",
            "Darf beeinflussen": "nur abschließende Schlussfolgerung",
            "Darf nicht beeinflussen": "Feature- oder Modellauswahl",
        },
    ]
)

display(experiment_blocks.set_index("Block"))


,Jahre,Darf beeinflussen,Darf nicht beeinflussen
Block,,,
Training,2015–2017,"Preprocessing, Modellparameter, Baselines",keine Einschränkung innerhalb des Trainings
Validierung,2018,"Featurewahl, Modellklasse, Hyperparameter",finale Testkennzahl
Test,2019,nur abschließende Schlussfolgerung,Feature- oder Modellauswahl


## 7. Baselines, Modelle und Featuregruppen

### Baselines

1. **Länderspezifischer Mittelwert:** bewusst triviale Untergrenze.
2. **Kalender-Baseline:** zentraler Vergleichsmaßstab aus
   Land × Monat × Werktagsklasse × Stunde mit dokumentierter
   Rückfallhierarchie bei kleinen Gruppen.

Beide Baselines werden ausschließlich aus dem Trainingsblock berechnet.

### Modellfamilien

- regularisierte lineare Regression als interpretierbarer Vergleich,
- Gradient Boosting als nichtlineares, tabellarisches Modell.

Die endgültige Modellklasse und ihre Hyperparameter werden ausschließlich auf
2018 gewählt.

### Kernfeatures

- Land,
- lokale Stunde, Wochentag, Wochenende, Monat und Jahreszeit,
- zyklische Sinus-/Kosinus-Kodierungen,
- Temperatur und nichtlineare Temperaturwirkung,
- ausreichend vollständige Strahlungsmerkmale.

Unmittelbare Last-Lags wie `lag_24h`, `lag_48h` oder `lag_168h` gehören nicht
zum Kernmodell. Ein separates lag-basiertes Day-ahead-Modell wäre ein anderer,
optionaler Anwendungsfall.


## 8. Leakage-Schutz

Folgende Regeln sind verbindlich:

- keine zufällige Aufteilung der Zeitreihe,
- keine Lastwerte aus 2018 oder 2019 beim Fit auf 2015–2017,
- Preprocessing nur auf Trainingsdaten fitten,
- Kalender-Baseline nur aus Trainingsdurchschnitten bilden,
- Hyperparameter nur anhand von 2018 wählen,
- 2019 erst nach dem Einfrieren der Konfiguration auswerten,
- keine TSO-Lastprognose als Eingabefeature,
- Reanalysewetter im historischen Backtest nicht als echte
  Day-ahead-Wetterprognose bezeichnen,
- klimatologisches Wetterprofil der App nicht mit tatsächlichem Testwetter
  verwechseln.


## 9. App-Konzept und Wissenstransfer

### Modus 1 – Historischer Backtest

- Land und Datum aus dem Testzeitraum auswählen,
- ML-Prognose, beide Baselines und tatsächliche Last überlagern,
- Fehlerkennzahlen und Abweichungen anzeigen.

### Modus 2 – Zukunftsszenario

- Land und frei wählbares Datum auswählen,
- Kalenderfeatures automatisch erzeugen,
- statistisch typisches Wetterprofil einsetzen,
- Temperaturabweichung, allgemeine Nachfrageänderung und additive
  Rechenzentrumslast verändern,
- Basisszenario und verändertes Szenario vergleichen.

Eine dargestellte 24-Stunden-Kurve beschreibt den gewählten Tag. Sie ist kein
behaupteter operativer Prognosehorizont.

### Vorgesehene Ausgaben

1. prognostizierte Lastkurve,
2. tatsächliche Last und Baselines im Backtest,
3. Spitzenlastdifferenz in MW und Prozent,
4. Wahrscheinlichkeit eines extremen Lastzustands relativ zu einer
   landesspezifischen historischen Quantilschwelle.

Die letzte Kennzahl ist ausdrücklich keine Blackout- oder
Netzausfallwahrscheinlichkeit.


## 10. Risiken, Grenzen und Gegenmaßnahmen

| Risiko / Grenze | Gegenmaßnahme |
|---|---|
| Zeitliche Datenverschiebung | streng chronologischer Backtest und Fehleranalyse je Jahr/Land |
| Dominanz großer Länder in MW-Metriken | länderweiser nMAE und gleichgewichtetes Makromittel |
| Wetter ist Reanalyse statt Vorhersage | im Backtest und Vortrag ausdrücklich kennzeichnen |
| Kurzer Lastzeitraum | keine sichere Extrapolation jahrzehntelanger Trends |
| Nationale Wetteraggregate glätten Extreme | als Datenlimit dokumentieren |
| Drei Länder repräsentieren nicht ganz Europa | Scope offen benennen; Erweiterung nur nach Qualitätsfreigabe |
| Szenarien werden als Prognosen missverstanden | konditionale Annahmen, Basisszenario und Regler klar beschriften |
| Extremzustand wird mit Blackout verwechselt | Quantilschwelle, Referenzperiode und Kalibrierung immer anzeigen |


## 11. Nachgelagerte U-Rückmeldung

Dieser Abschnitt berichtet transparent, was die **später ausgeführte U-Phase**
zur Datenhypothese H1 ergab. Er ist keine rückwirkende Änderung der
Forschungsfrage.

- 131.472 mögliche Länder-Stunden für DE, FR und PL,
- 131.441 nutzbare Länder-Stunden,
- 31 fehlende Lastwerte (0,024 %), die nicht imputiert werden,
- vollständige Wetterfeatures im Modellfenster 2015–2019,
- keine doppelten Länder-Zeit-Schlüssel,
- keine Lücken in den UTC-Zeitachsen,
- H1 ist damit bestätigt.

Die vollständige Analyse steht in
[`01_data_import_merge_eda.ipynb`](01_data_import_merge_eda.ipynb).


In [6]:
u_feedback = {
    "possible_country_hours": 131_472,
    "usable_country_hours": 131_441,
    "missing_target_values": 31,
    "weather_complete": True,
    "duplicate_country_timestamps": 0,
}

u_feedback["target_completeness_pct"] = (
    100
    * u_feedback["usable_country_hours"]
    / u_feedback["possible_country_hours"]
)
u_feedback["H1_confirmed"] = (
    u_feedback["target_completeness_pct"] >= 95
    and u_feedback["weather_complete"]
    and u_feedback["duplicate_country_timestamps"] == 0
)

display(pd.Series(u_feedback, name="U-Ergebnis").to_frame())


,U-Ergebnis
possible_country_hours,131472
usable_country_hours,131441
missing_target_values,31
weather_complete,True
duplicate_country_timestamps,0
target_completeness_pct,99.976421
H1_confirmed,True


## 12. Offene Entscheidungen für A³ und C

Die Q-Phase legt nicht das spätere Ergebnis vorweg. Offen bleiben:

- genaue Regularisierungs- und Boosting-Hyperparameter,
- endgültige Auswahl der Wettermerkmale,
- Nutzen optionaler Feiertagsfeatures,
- Gewinner der Modellfamilien auf 2018,
- tatsächlicher Mehrwert gegenüber der Kalender-Baseline auf 2019,
- Fehlerunterschiede zwischen Ländern, Tageszeiten und Saisons,
- Kalibrierung der Extremzustandswahrscheinlichkeit.

Diese Punkte dürfen erst durch die dafür vorgesehenen Phasen entschieden
werden.


In [7]:
checks = {
    "Drei freigegebene Kernländer": set(PROJECT_SPEC["countries"]) == {"DE", "FR", "PL"},
    "Chronologische Blockreihenfolge": (
        PROJECT_SPEC["train"].startswith("2015")
        and PROJECT_SPEC["validation"].startswith("2018")
        and PROJECT_SPEC["test"].startswith("2019")
    ),
    "Zwei Baselines spezifiziert": len(PROJECT_SPEC["baselines"]) == 2,
    "Mindestens zwei Modellfamilien": len(PROJECT_SPEC["model_families"]) >= 2,
    "Makro-nMAE als Primärmetrik": PROJECT_SPEC["primary_metric"] == "Makro-nMAE",
    "Keine Last-Lags im Kernmodell": PROJECT_SPEC["core_uses_load_lags"] is False,
    "Datenhypothese H1 nach U bestätigt": u_feedback["H1_confirmed"] is True,
}

assert all(checks.values()), [name for name, passed in checks.items() if not passed]

checks_table = pd.DataFrame(
    {
        "Konsistenzprüfung": list(checks),
        "Status": ["Bestanden" if passed else "Fehlgeschlagen" for passed in checks.values()],
    }
)
display(checks_table.set_index("Konsistenzprüfung"))
display(Markdown(f"**Ergebnis:** {sum(checks.values())}/{len(checks)} Prüfungen bestanden."))


,Status
Konsistenzprüfung,
Drei freigegebene Kernländer,Bestanden
Chronologische Blockreihenfolge,Bestanden
Zwei Baselines spezifiziert,Bestanden
Mindestens zwei Modellfamilien,Bestanden
Makro-nMAE als Primärmetrik,Bestanden
Keine Last-Lags im Kernmodell,Bestanden
Datenhypothese H1 nach U bestätigt,Bestanden


**Ergebnis:** 7/7 Prüfungen bestanden.

## 13. Abschluss der Q-Phase

Die Q-Phase ist abgeschlossen, wenn:

- Forschungsfragen und Zielgruppen eindeutig sind,
- Hypothesen und Entscheidungsregeln vor dem Modellvergleich feststehen,
- Scope, Metriken, Baselines und Split dokumentiert sind,
- Backtest und Zukunftsszenario klar getrennt sind,
- Grenzen und Nicht-Ziele benannt sind.

Diese Bedingungen sind erfüllt. Die U-Phase hat die grundsätzliche
Daten-Eignung bestätigt. Als nächster Schritt folgt:

> **A³ – Algorithm Selection, Adapting Features, Adjusting Hyperparameters**

Dort werden Baselines und Regressionsmodelle ausschließlich mit 2015–2017
trainiert und anhand von 2018 verglichen. Das Testjahr 2019 bleibt bis zur
abschließenden C-Phase unangetastet.
